# PUENTE — LoRA Fine-Tuning for Chavacano Translation

Train Low-Rank Adaptation (LoRA) adapters on top of **facebook/nllb-200-distilled-600M**
for formal and street (colloquial) Chavacano translation.

## Target Languages (FLORES-200)
| Code | FLORES | Language |
|------|--------|----------|
| en   | eng_Latn | English (pivot) |
| tl   | tgl_Latn | Tagalog |
| cbk  | cbk_Latn | Chavacano de Zamboanga |
| hil  | hil_Latn | Hiligaynon |
| ceb  | ceb_Latn | Cebuano / Bisaya |

## Steps
1. Load base NLLB-200 model (8-bit quantized)
2. Prepare parallel sentence dataset
3. Configure LoRA (r=16, alpha=32, q_proj + v_proj)
4. Train with AdamW optimizer
5. Save adapter weights to `ml_models/lora-cbk-{mode}/`

In [ ]:
# Cell 1: Install dependencies (run once)
# Uncomment and run if packages are not yet installed

# !pip install torch transformers sentencepiece accelerate peft bitsandbytes

In [ ]:
# Cell 2: Imports and Configuration
import json
import os
import time

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

# ── Configuration ──────────────────────────────────────────────
BASE_MODEL_DIR = '../ml_models/nllb-200-distilled-600M'
MODE = 'formal'  # Change to 'street' for colloquial adapter
OUTPUT_DIR = f'../ml_models/lora-cbk-{MODE}'

# Training hyperparameters
EPOCHS = 3
BATCH_SIZE = 4
LEARNING_RATE = 2e-4
MAX_LENGTH = 128

# LoRA hyperparameters
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ['q_proj', 'v_proj']

# FLORES codes
SRC_LANG = 'eng_Latn'   # English as source
TGT_LANG = 'cbk_Latn'   # Chavacano as target

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Mode:   {MODE}')
print(f'Output: {OUTPUT_DIR}')

In [ ]:
# Cell 3: Load Base Model + Tokenizer
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR)

print('Loading base model...')
try:
    from transformers import BitsAndBytesConfig
    quant_config = BitsAndBytesConfig(load_in_8bit=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        BASE_MODEL_DIR,
        quantization_config=quant_config,
        device_map='auto',
    )
    print('Loaded with 8-bit quantization')
except Exception as e:
    print(f'8-bit not available ({e}), loading in FP32...')
    model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL_DIR).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Base model params: {total_params:,}')

In [ ]:
# Cell 4: Prepare Training Data
#
# Expected JSON format (array of parallel sentence pairs):
# [
#   {"source": "Good morning", "target": "Buenos días"},
#   {"source": "Thank you",    "target": "Gracias"},
#   ...
# ]

DATASET_PATH = '../Datasets/processed/001_chavacano/chavacano_parallel_sentences_nllb.json'

if os.path.exists(DATASET_PATH):
    with open(DATASET_PATH, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    print(f'Loaded {len(raw_data)} parallel pairs from dataset')
else:
    # Demo data for testing the pipeline
    raw_data = [
        {'source': 'Good morning', 'target': 'Buenos días'},
        {'source': 'How are you?', 'target': 'Como está usted?'},
        {'source': 'Thank you very much', 'target': 'Muchas gracias'},
        {'source': 'I love you', 'target': 'Ta ama yo contigo'},
        {'source': 'Good evening', 'target': 'Buenas noches'},
        {'source': 'Where is the market?', 'target': 'Donde el mercado?'},
        {'source': 'The food is delicious', 'target': 'El comida ta sabroso'},
        {'source': 'See you tomorrow', 'target': 'Hasta mañana'},
    ]
    print(f'Using {len(raw_data)} demo pairs (dataset not found at {DATASET_PATH})')

# Tokenize
def tokenize_pairs(pairs, tokenizer, src_lang, tgt_lang, max_length=128):
    """Tokenize parallel sentence pairs for NLLB-200 training."""
    tokenizer.src_lang = src_lang
    sources = [p['source'] for p in pairs]
    targets = [p['target'] for p in pairs]
    
    model_inputs = tokenizer(
        sources, max_length=max_length, truncation=True, padding=True, return_tensors='pt'
    )
    
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets, max_length=max_length, truncation=True, padding=True, return_tensors='pt'
        )
    
    model_inputs['labels'] = labels['input_ids']
    # Replace padding tokens with -100 so they're ignored in loss
    model_inputs['labels'][model_inputs['labels'] == tokenizer.pad_token_id] = -100
    return model_inputs

train_data = tokenize_pairs(raw_data, tokenizer, SRC_LANG, TGT_LANG, MAX_LENGTH)
print(f'Tokenized: input_ids shape = {train_data["input_ids"].shape}')
print(f'           labels shape    = {train_data["labels"].shape}')

In [ ]:
# Cell 5: Configure LoRA Adapter
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias='none',
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)')
model.print_trainable_parameters()

In [ ]:
# Cell 6: Training Loop
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset

# Create DataLoader
dataset = TensorDataset(
    train_data['input_ids'],
    train_data['attention_mask'],
    train_data['labels'],
)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
model.train()

print(f'Training {EPOCHS} epochs, {len(dataloader)} batches/epoch, lr={LEARNING_RATE}')
print('=' * 60)

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    start = time.perf_counter()
    
    for batch_idx, (input_ids, attention_mask, labels) in enumerate(dataloader):
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        epoch_loss += loss.item()
    
    elapsed = time.perf_counter() - start
    avg_loss = epoch_loss / len(dataloader)
    print(f'  Epoch {epoch + 1}/{EPOCHS}  loss={avg_loss:.4f}  time={elapsed:.1f}s')

print('\nTraining complete!')

In [ ]:
# Cell 7: Save LoRA Adapter
os.makedirs(OUTPUT_DIR, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
print(f'LoRA adapter saved to: {OUTPUT_DIR}')

# Verify saved files
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f'  {f} ({size:,} bytes)')

In [ ]:
# Cell 8: Quick Validation — Test the Fine-Tuned Model
from peft import PeftModel

# Reload base model + merge adapter
print('Reloading base model for validation...')
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL_DIR).to(device)
finetuned = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
finetuned = finetuned.merge_and_unload()
finetuned.eval()
print('Adapter merged successfully')

# Test translations
test_cases = [
    ('Good morning', 'eng_Latn', 'cbk_Latn'),
    ('How are you?', 'eng_Latn', 'cbk_Latn'),
    ('Thank you very much', 'eng_Latn', 'cbk_Latn'),
    ('I love this city', 'eng_Latn', 'cbk_Latn'),
]

print(f'\n--- {MODE.upper()} Adapter Translations ---')
for text, src, tgt in test_cases:
    tokenizer.src_lang = src
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128).to(device)
    tgt_id = tokenizer.convert_tokens_to_ids(tgt)
    
    with torch.no_grad():
        output = finetuned.generate(
            **inputs,
            forced_bos_token_id=tgt_id,
            max_new_tokens=128,
            num_beams=4,
        )
    result = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
    print(f'  {text:<30} → {result}')

print(f'\nDone! Adapter at: {OUTPUT_DIR}')